In [5]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import joblib
pd.set_option("display.max_columns",None)

In [6]:
df = pd.read_csv(r"D:\SecurePay-AI\dataset\cleaned\transactions_clean.csv")

print("shape :" , df.shape)
df.head()

shape : (6362620, 11)


,step,type,amount,nameorig,oldbalanceorg,newbalanceorig,namedest,oldbalancedest,newbalancedest,isfraud,isflaggedfraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameorig        object 
 4   oldbalanceorg   float64
 5   newbalanceorig  float64
 6   namedest        object 
 7   oldbalancedest  float64
 8   newbalancedest  float64
 9   isfraud         int64  
 10  isflaggedfraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [8]:
required_columns = [
    "step",
    "type",
    "amount",
    "nameorig",
    "oldbalanceorg",
    "newbalanceorig",
    "namedest",
    "oldbalancedest",
    "newbalancedest",
    "isfraud"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("missing columns :",missing_columns)

missing columns : []


In [9]:
# sort transaction by time 
df = df.sort_values(
    by=["step"]
).reset_index(drop=True)

df[["step", "nameorig", "namedest", "amount", "isfraud"]].head(10)

,step,nameorig,namedest,amount,isfraud
0,1,C1231006815,M1979787155,9839.64,0
1,1,C1666544295,M2044282225,1864.28,0
2,1,C1305486145,C553264065,181.00,1
3,1,C840083671,C38997010,181.00,1
4,1,C2048537720,M1230701703,11668.14,0
5,1,C90045638,M573487274,7817.71,0
6,1,C154988899,M408069119,7107.77,0
7,1,C1912850431,M633326333,7861.64,0
8,1,C1265012928,M1176932104,4024.36,0
9,1,C712410124,C195600860,5337.77,0


In [10]:
# creating historical receiver feature 
df["receiver_trnsaction_count"]=(
    df.groupby("namedest").cumcount()
)

In [11]:
df[
    [
        "namedest",
        "receiver_trnsaction_count"
    ]
].head(10)

,namedest,receiver_trnsaction_count
0,M1979787155,0
1,M2044282225,0
2,C553264065,0
3,C38997010,0
4,M1230701703,0
5,M573487274,0
6,M408069119,0
7,M633326333,0
8,M1176932104,0
9,C195600860,0


In [12]:
# Check the exact column name
# print([
#     col for col in df.columns
#     if "receiver" in col.lower()
# ])
df.rename(
    columns={
        "receiver_trnsaction_count": "receiver_transaction_count"
    },
    inplace=True
)

In [13]:
# PREVIOUS RECEIVER FRAUD COUNT
df["previous_receiver_fraud"] = (
    df.groupby("namedest")["isfraud"].transform(lambda x: x.shift(1).fillna(0).cumsum())
)

In [14]:
df[["namedest","isfraud","receiver_transaction_count","previous_receiver_fraud"]].head(10)

,namedest,isfraud,receiver_transaction_count,previous_receiver_fraud
0,M1979787155,0,0,0.0
1,M2044282225,0,0,0.0
2,C553264065,1,0,0.0
3,C38997010,1,0,0.0
4,M1230701703,0,0,0.0
5,M573487274,0,0,0.0
6,M408069119,0,0,0.0
7,M633326333,0,0,0.0
8,M1176932104,0,0,0.0
9,C195600860,0,0,0.0


In [15]:
# historical receiver fraud rate
df["historical_receiver_fraud_rate"] = np.where(
    df["receiver_transaction_count"] > 0,
    df["previous_receiver_fraud"] /
    df["receiver_transaction_count"],
    0
)

In [16]:
df[
    [
        "namedest",
        "receiver_transaction_count",
        "previous_receiver_fraud",
        "historical_receiver_fraud_rate"
    ]
].head(20)

,namedest,receiver_transaction_count,previous_receiver_fraud,historical_receiver_fraud_rate
0,M1979787155,0,0.0,0.0
1,M2044282225,0,0.0,0.0
2,C553264065,0,0.0,0.0
3,C38997010,0,0.0,0.0
4,M1230701703,0,0.0,0.0
5,M573487274,0,0.0,0.0
6,M408069119,0,0.0,0.0
7,M633326333,0,0.0,0.0
8,M1176932104,0,0.0,0.0
9,C195600860,0,0.0,0.0


In [17]:
# previous receiver amount 
df["previous_receiver_amount"] = (
    df.groupby("namedest")["amount"].transform(lambda x:x.shift(1).fillna(0).cumsum())
)

In [18]:
df[["namedest","amount","previous_receiver_amount"]].head(20)

,namedest,amount,previous_receiver_amount
0,M1979787155,9839.64,0.0
1,M2044282225,1864.28,0.0
2,C553264065,181.00,0.0
3,C38997010,181.00,0.0
4,M1230701703,11668.14,0.0
5,M573487274,7817.71,0.0
6,M408069119,7107.77,0.0
7,M633326333,7861.64,0.0
8,M1176932104,4024.36,0.0
9,C195600860,5337.77,0.0


In [19]:
# historical sender transaction count 

df["sender_transaction_count"] =(
    df.groupby("nameorig").cumcount()
)

In [20]:
df[["nameorig","sender_transaction_count"]].head(20)

,nameorig,sender_transaction_count
0,C1231006815,0
1,C1666544295,0
2,C1305486145,0
3,C840083671,0
4,C2048537720,0
5,C90045638,0
6,C154988899,0
7,C1912850431,0
8,C1265012928,0
9,C712410124,0


In [21]:
# historical sender amount 

df["previous_sender_amount"] = (
    df.groupby("nameorig")["amount"].transform(lambda x: x.shift(1).fillna(0).cumsum)
)

In [22]:
df[["nameorig","amount","previous_sender_amount"]].head(20)

,nameorig,amount,previous_sender_amount
0,C1231006815,9839.64,<bound method Series.cumsum of 0 0.0\nName:...
1,C1666544295,1864.28,<bound method Series.cumsum of 1 0.0\nName:...
2,C1305486145,181.00,<bound method Series.cumsum of 2 0.0\nName:...
3,C840083671,181.00,<bound method Series.cumsum of 3 0.0\nName:...
4,C2048537720,11668.14,<bound method Series.cumsum of 4 0.0\nName:...
5,C90045638,7817.71,<bound method Series.cumsum of 5 0.0\nName:...
6,C154988899,7107.77,<bound method Series.cumsum of 6 0.0\nName:...
7,C1912850431,7861.64,<bound method Series.cumsum of 7 0.0\nName:...
8,C1265012928,4024.36,<bound method Series.cumsum of 8 0.0\nName:...
9,C712410124,5337.77,<bound method Series.cumsum of 9 0.0\nName:...
